In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft, fftfreq
from scipy.signal.windows import hann
import pickle as pkl
import os
from scipy.signal import welch
from xgboost import XGBClassifier
import seaborn as sns
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score, balanced_accuracy_score
import secrets
from sklearn.model_selection import train_test_split

import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 150

# Data Load

In [ ]:
data=np.load('data/photostim_steps.npy')
description=pd.read_csv('data/photostim_steps.csv')

print(data.shape)
print(len(description))

channels = ['FP1', 'F3', 'C3', 'P3', 'F7', 'T3', 'T5', 'O1', 'FZ', 'CZ',
            'PZ', 'FP2', 'F4', 'C4', 'P4', 'F8', 'T4', 'T6', 'O2']

N_FFT = 16384
FS=250

spectra = []

for i, sample in enumerate(data):
    sample_specs=[]
    for j,channel in enumerate(sample[:19]):
        channel_hann = channel*hann(len(channel))
        ch_fft = fft(channel_hann,N_FFT)[:N_FFT//2]
        #ch_fft = welch(channel,fs=FS,nperseg=N_FFT, nfft=N_FFT)[1][:-1]
        ch_fft=abs(ch_fft)/np.sum(abs(ch_fft))

        #print(f'Sample {i} - Channel {channels[j]} - Total power: {np.sum(abs(ch_fft))}')

        sample_specs.append(ch_fft)
    spectra.append(sample_specs)

spectra = np.array(spectra)
print(spectra.shape)

freqaxis=fftfreq(N_FFT,1/FS)[:N_FFT//2]

freq_bins= np.arange(0, 125,2)

In [ ]:
# bin the spectra in 2 hz bins
binned_spectra = []

bin_size = 132

for sample in spectra:
    binned_spectra_sample=[]
    for j,channel in enumerate(sample[:19]):
        binned_spectrum=[]
        n_bins = len(channel)//bin_size
        binned_amplitudes = channel[:n_bins * bin_size].reshape(-1, bin_size).sum(axis=1)
        binned_spectra_sample.append(binned_amplitudes)
        #print(f'Channel {channels[j]} - Total power: {np.sum(binned_amplitudes)}')
    binned_spectra.append(binned_spectra_sample)
binned_spectra = np.array(binned_spectra)
print(binned_spectra.shape)

# Spectra Plots

## Average Spectrum - All frequency steps

In [ ]:
epileptic_idxs=np.where(description['epilepsy']==1)
control_idxs=np.where(description['epilepsy']==0)

epileptic_spectra = spectra[epileptic_idxs]
control_spectra = spectra[control_idxs]

print(f'epilepsy_spectra shape: {epileptic_spectra.shape}')
print(f'control_spectra shape: {control_spectra.shape}')

avg_epileptic_spectra=np.mean(epileptic_spectra, axis=0)
avg_control_spectra=np.mean(control_spectra, axis=0)

plt.figure(figsize=(20,15))
for i, channel in enumerate(channels):
    plt.subplot(5,4,i+1)
    plt.plot(freqaxis, avg_epileptic_spectra[i], label='epilepsy', alpha=0.5, color='red')
    plt.plot(freqaxis, avg_control_spectra[i], label='healthy', alpha=0.5, color='green')
    plt.title(channel)
    plt.xlim(0,30)
    plt.yscale('log')
    plt.legend()
plt.suptitle('Spectra of epilepsy and control groups during photostimulation', fontsize=16)
plt.tight_layout()
plt.show()

## Average Binned Spectrum - All frequency steps

In [ ]:
epileptic_idxs=np.where(description['epilepsy']==1)
control_idxs=np.where(description['epilepsy']==0)

epileptic_spectra_binned = binned_spectra[epileptic_idxs]
control_spectra_binned = binned_spectra[control_idxs]

print(f'epilepsy_spectra shape: {epileptic_spectra_binned.shape}')
print(f'control_spectra shape: {control_spectra_binned.shape}')

avg_epileptic_spectra_binned=np.mean(epileptic_spectra_binned, axis=0)
avg_control_spectra_binned=np.mean(control_spectra_binned, axis=0)

ff=np.arange(1,125,2)

plt.figure(figsize=(20,15))
for i, channel in enumerate(channels):
    plt.subplot(5,4,i+1)
    plt.bar(ff, avg_epileptic_spectra_binned[i], label='epilepsy', alpha=0.5, width=2, color='red')
    plt.bar(ff, avg_control_spectra_binned[i], label='healthy', alpha=0.5, width=2, color='green')
    plt.title(channel)
    plt.xlim(0,50)
    plt.yscale('log')
    plt.legend()
plt.suptitle('Binned spectra of epilepsy and control groups during photostimulation', fontsize=18)
plt.tight_layout()
plt.show()

## Average Spectrum per frequency step

In [ ]:
freqs=np.arange(1,23,2)

freq_idxs=[]

for freq in freqs:
    idx_at_freq=description[description['frequency']==freq].index.tolist()
    print(f'Frequency: {freq} Hz')
    print(f'Number of samples: {len(idx_at_freq)}')
    freq_idxs.append(idx_at_freq)

print()

for i, idxs in enumerate(freq_idxs):
    print('Frequency: ', freqs[i])

    epileptic_idxs_at_freq=np.where(description.loc[idxs]['epilepsy']==1)
    control_idxs_at_freq=np.where(description.loc[idxs]['epilepsy']==0)

    print(f'Epilepsy: {len(epileptic_idxs_at_freq[0])}')
    print(f'Control: {len(control_idxs_at_freq[0])}')

    epileptic_spectra_at_freq=spectra[epileptic_idxs_at_freq]
    control_spectra_at_freq=spectra[control_idxs_at_freq]

    avg_epileptic_spectra_at_freq=np.mean(epileptic_spectra_at_freq, axis=0)
    avg_control_spectra_at_freq=np.mean(control_spectra_at_freq, axis=0)
   
    plt.figure(figsize=(20,15))
    for j, channel in enumerate(channels):
        plt.subplot(5,4,j+1)
        plt.plot(freqaxis,avg_epileptic_spectra_at_freq[j], label='epilepsy', alpha=0.5, color='red')
        plt.plot(freqaxis,avg_control_spectra_at_freq[j], label='control', alpha=0.5, color='green')
        plt.title(channel)
        plt.xticks(np.arange(0,125,freqs[i]))
        plt.xlim(0,50)
        plt.yscale('log')
        plt.legend()
    plt.suptitle('Stimulation Frequency: '+str(freqs[i]), fontsize=18)
    plt.tight_layout()
    plt.show()

## Average binned spectrum per frequency step

In [ ]:
#divide the spectra based on the epilepsy label in the description for each stimulation frequency
for i, idxs in enumerate(freq_idxs):
    print('Frequency: ', freqs[i])

    epileptic_idxs_at_freq=np.where(description.loc[idxs]['epilepsy']==1)
    control_idxs_at_freq=np.where(description.loc[idxs]['epilepsy']==0)

    print(f'Epilepsy: {len(epileptic_idxs_at_freq[0])}')
    print(f'Control: {len(control_idxs_at_freq[0])}')

    epileptic_spectra_at_freq_binned=binned_spectra[epileptic_idxs_at_freq]
    control_spectra_at_freq_binned=binned_spectra[control_idxs_at_freq]

    avg_epileptic_spectra_at_freq_binned=np.mean(epileptic_spectra_at_freq_binned, axis=0)
    avg_control_spectra_at_freq_binned=np.mean(control_spectra_at_freq_binned, axis=0)
    
    plt.figure(figsize=(20,20))
    for j, channel in enumerate(channels):
        plt.subplot(5,4,j+1)
        plt.plot(ff,avg_epileptic_spectra_at_freq_binned[j], label='epilepsy', alpha=0.5,color='red')
        plt.plot(ff,avg_control_spectra_at_freq_binned[j], label='control', alpha=0.5,  color='green')
        plt.title(channel)
        plt.xticks(ff)
        plt.xlim(0,25)
        plt.yscale('log')
        plt.legend()
    plt.suptitle('Stimulation Frequency: '+str(freqs[i]), fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
print(spectra.shape)
print(description.shape)
print(binned_spectra.shape)

# Save everything
np.save('data/spectra.npy', spectra)
np.save('data/binned_spectra.npy', binned_spectra)


## Frequency bins distribution - Healthy vs Epi

In [ ]:
# boxplot of the first 12 bins per channel for epilepsy vs healthy
plt.figure(figsize=(30,30))
for i, channel in enumerate(channels):
    plt.subplot(5,4,i+1)
    bp1 = plt.boxplot(epileptic_spectra_binned[:,i,:], positions=np.arange(0,124,2), widths=0.6, patch_artist=True, notch=True)
    bp2 = plt.boxplot(control_spectra_binned[:,i,:], positions=np.arange(1,125,2), widths=0.6, patch_artist=True, notch=True)
    
    for box in bp1['boxes']:
        box.set(color='red', linewidth=2)
        box.set(facecolor='red')
    
    for box in bp2['boxes']:
        box.set(color='green', linewidth=2)
        box.set(facecolor='green')
    
    for median in bp1['medians']:
        median.set(color='white', linewidth=2)
    
    for median in bp2['medians']:
        median.set(color='white', linewidth=2)

    plt.title(channel)
    plt.xlim(0,50)
    plt.yscale('log')
    plt.xlabel('Bin Centre Frequency (Hz)')
    plt.legend([bp1['boxes'][0], bp2['boxes'][0]], ['epilepsy', 'control'])
    plt.xticks(ticks=np.arange(0,50,2),labels=np.arange(1,50,2))

plt.suptitle('Binned spectra of epilepsy and control groups during photostimulation', fontsize=18)
plt.tight_layout()
plt.show()


### Test f-boxplots

In [ ]:
from statsmodels.graphics.functional import fboxplot
epi_subjects=description.loc[epileptic_idxs][['subject']].to_numpy()
print(epi_subjects.shape)

In [ ]:
fig = plt.figure(figsize=(50,50))

for i, channel in enumerate(channels):
    ax=fig.add_subplot(5,4,i+1)
    res = fboxplot(data=epileptic_spectra[:,i,:], xdata=ff, ax = ax , method='BD2', labels=epi_subjects)
    plt.title(channel)
    plt.xlim(0,50)
    plt.yscale('log')
    plt.xlabel('Bin Centre Frequency (Hz)')
    plt.xticks(ticks=np.arange(0,50,2),labels=np.arange(1,50,2))

plt.show()

## Background spectra - No Photostim

In [ ]:
no_stim_data=pkl.load(open('data/no_photostim_periods.pkl','rb'))
no_stim_df=pd.read_csv('data/no_photostim.csv')
no_stim_spectra = []

for i, sample in enumerate(no_stim_data):
    sample_specs=[]
    for j,channel in enumerate(sample[:19]):
        channel_hann = channel*hann(len(channel))

        #ch_fft = fft(channel_hann,N_FFT)[:N_FFT//2]
        ch_fft = welch(channel, fs=FS, nperseg=N_FFT, nfft=N_FFT, scaling='spectrum')[1][:-1]
        ch_fft=abs(ch_fft)/np.sum(abs(ch_fft))
        #print(f'Channel {channels[j]} - Total power: {np.sum(abs(ch_fft))}')
        sample_specs.append(ch_fft)
    no_stim_spectra.append(sample_specs)

no_stim_spectra = np.array(no_stim_spectra)
print(no_stim_spectra.shape)

no_stim_spectra_binned = []
bin_size = 132

for sample in no_stim_spectra:
    binned_spectra_sample=[]
    for channel in sample:
        binned_spectrum=[]
        n_bins = len(channel)//bin_size
        binned_amplitudes = channel[:n_bins * bin_size].reshape(-1, bin_size).sum(axis=1)
        #print(f'Total power: {np.sum(binned_amplitudes)}')
        binned_spectra_sample.append(binned_amplitudes)

    no_stim_spectra_binned.append(binned_spectra_sample)

no_stim_spectra_binned = np.array(no_stim_spectra_binned)
print(no_stim_spectra_binned.shape)

## Background spectra - No Photostim

In [ ]:
no_stim_healthy_idxs=np.where(no_stim_df['epilepsy']==0)
no_stim_epileptic_idxs=np.where(no_stim_df['epilepsy']==1)

no_stim_healthy_spectra = no_stim_spectra[no_stim_healthy_idxs]
no_stim_epileptic_spectra = no_stim_spectra[no_stim_epileptic_idxs]

avg_no_stim_healthy_spectra=np.mean(no_stim_healthy_spectra, axis=0)
avg_no_stim_epileptic_spectra=np.mean(no_stim_epileptic_spectra, axis=0)

plt.figure(figsize=(30,30))
for i, channel in enumerate(channels):
    plt.subplot(5,4,i+1)
    plt.plot(freqaxis, avg_no_stim_healthy_spectra[i], label='healthy - background', alpha=0.5, color='green')
    plt.plot(freqaxis, avg_no_stim_epileptic_spectra[i], label='epilepsy - background ', alpha=0.5, color='red')
    plt.plot(freqaxis, avg_control_spectra[i], label='healthy - stimulation', alpha=0.5, color='green', linestyle='--')
    plt.plot(freqaxis, avg_epileptic_spectra[i], label='epilepsy - stimulation', alpha=0.5, color='red', linestyle='--')
    plt.title(channel)
    plt.xlim(0,35)
    plt.ylim(1e-5, 1)
    plt.yscale('log')
    plt.legend()

## Background spectra binned - No Photostim

In [ ]:
no_stim_healthy_spectra_binned=no_stim_spectra_binned[no_stim_healthy_idxs]
no_stim_epileptic_spectra_binned=no_stim_spectra_binned[no_stim_epileptic_idxs]

print(no_stim_healthy_spectra_binned.shape)
print(no_stim_epileptic_spectra_binned.shape)

avg_no_stim_healthy_spectra_binned=np.mean(no_stim_healthy_spectra_binned, axis=0)
avg_no_stim_epileptic_spectra_binned=np.mean(no_stim_epileptic_spectra_binned, axis=0)

#plot healthy and epilepsy spectra for no stimulation periods
plt.figure(figsize=(20,15))
for i, channel in enumerate(channels):
    plt.subplot(5,4,i+1)
    plt.plot(ff, avg_no_stim_epileptic_spectra_binned[i], label='epilepsy - background', alpha=0.5, color='red')
    plt.plot(ff, avg_no_stim_healthy_spectra_binned[i], label='healthy - background', alpha=0.5, color='green')
    plt.plot(ff, avg_epileptic_spectra_binned[i], label='epilepsy - stimulation', alpha=0.5, color='red', linestyle='--')
    plt.plot(ff, avg_control_spectra_binned[i], label='healthy - stimulation', alpha=0.5, color='green', linestyle='--')
    plt.title(channel)
    plt.xlim(0,50)
    plt.yscale('log')
    plt.legend()
plt.suptitle('Spectra of epilepsy and control groups during no stimulation periods')
plt.tight_layout()
plt.show()

## Subject based spectra

In [ ]:
unique_subjects=np.unique(description['subject'])

for subject in unique_subjects:
    subject_idxs_stim=np.where(description['subject']==subject)
    subject_spectra_stim=spectra[subject_idxs_stim]
    avg_subject_spectra_stim=np.mean(subject_spectra_stim, axis=0)
    subject_idxs_nostim=np.where(no_stim_df['subject']==subject)
    subject_spectra_no_stim=no_stim_spectra[subject_idxs_nostim]
    avg_subject_spectra_no_stim=np.mean(subject_spectra_no_stim, axis=0)
    plt.figure(figsize=(20,15))
    for i, channel in enumerate(channels):
        plt.subplot(5,4,i+1)
        plt.plot(freqaxis, avg_subject_spectra_stim[i], label='Photostimulation', alpha=0.5, linestyle='--')
        plt.plot(freqaxis, avg_subject_spectra_no_stim[i], label='No Stimulation', alpha=0.5)
        plt.title(channel)
        plt.xlim(0,50)
        plt.yscale('log')
        plt.legend()
    plt.suptitle('Subject: '+subject)
    plt.tight_layout()
    plt.show()

## Subject based spectra Binned

In [ ]:
for subject in unique_subjects:
    subject_idxs_stim=np.where(description['subject']==subject)
    subject_spectra_stim=binned_spectra[subject_idxs_stim]
    avg_subject_spectra_stim=np.mean(subject_spectra_stim, axis=0)
    subject_idxs_nostim=np.where(no_stim_df['subject']==subject)
    subject_spectra_no_stim=no_stim_spectra_binned[subject_idxs_nostim]
    avg_subject_spectra_no_stim=np.mean(subject_spectra_no_stim, axis=0)
    plt.figure(figsize=(20,15))
    for i, channel in enumerate(channels):
        plt.subplot(5,4,i+1)
        plt.plot(ff, avg_subject_spectra_stim[i], label='Photostimulation', alpha=0.5, linestyle='--')
        plt.plot(ff, avg_subject_spectra_no_stim[i], label='No Stimulation', alpha=0.5)
        plt.title(channel)
        plt.xlim(0,50)
        plt.yscale('log')
        plt.legend()
    plt.suptitle(f'Subject: {subject} - Epilepsy: {description.loc[subject_idxs_stim[0][0]]["epilepsy"]}')
    plt.tight_layout()
    plt.show()

In [ ]:
labels=np.where(description['epilepsy']==1,1,0)

feats=[]
for i,spectrum in enumerate(binned_spectra):

    sample_feats=[]

    for channel in spectrum:
        for j, bin in enumerate(channel):
            if j == description.loc[i]['frequency']//2:
                sample_feats.append(bin)

    feats.append(sample_feats)

feats=np.array(feats)

print(feats.shape)


# XGBoost tests

## Binned Spectra - Stimulation

In [14]:
def make_splits(subjects, test_size=0.05):
    train_subjects, test_subjects = train_test_split(subjects, test_size=test_size)
    print(f'Number of train subjects: {len(train_subjects)}')
    print(f'Number of test subjects: {len(test_subjects)}')
    train_idxs = description[description['subject'].isin(train_subjects)].index
    test_idxs = description[description['subject'].isin(test_subjects)].index
    return train_idxs, test_idxs

In [30]:
def calculate_bac(labels, scores, sens_thresh):
    fpr, tpr, thresholds = roc_curve(labels, scores)
    threshold_sensitivity = thresholds[np.where(tpr >= sens_thresh)[0][0]]
    adjusted_predictions = (scores >= threshold_sensitivity).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, adjusted_predictions).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    bac = ((sensitivity + specificity) / 2) * 100
    return bac, fpr, tpr, thresholds

In [ ]:
unique_subjects=np.unique(description['subject'])

seed=secrets.randbelow(1000)
print(f'Seed: {seed}')

labels=np.where(description['epilepsy']==1,1,0)

y_groundtruths=[]
y_preds=[]
y_pred_probs=[]

for i,subject in enumerate(unique_subjects):

    test_idxs=np.where(description['subject']==subject)
    train_idxs=np.where(description['subject']!=subject)

    print(f'Iteration {i} - Subject left out: {subject}')
    print(f'Number of train samples: {len(train_idxs[0])}')
    print(f'Number of test samples: {len(test_idxs[0])}')

    x_train=binned_spectra[train_idxs][:,:20].reshape(len(train_idxs[0]),-1)
    x_test=binned_spectra[test_idxs][:,:20].reshape(len(test_idxs[0]),-1)
    y_train=labels[train_idxs]
    y_test=labels[test_idxs]

    ratio=(len(y_train)-sum(y_train))/sum(y_train)                   #len(np.where(y_train==0)[0])/len(np.where(y_train==1)[0])
    model=XGBClassifier(scale_pos_weight=ratio, n_jobs=4, device='cuda:0', n_estimators=100, seed=seed, max_depth=6, subsample=0.9, gamma=0.1, learning_rate=0.01)
    model.fit(x_train,y_train)
    y_pred=model.predict(x_test)
    y_pred_prob = model.predict_proba(x_test)[:,1]

    y_groundtruths.extend(y_test)
    y_preds.extend(y_pred)
    y_pred_probs.extend(y_pred_prob)

accuracy = accuracy_score(y_groundtruths, y_preds)
print("Accuracy: %.2f%%" % (accuracy * 100.0))
print("Balanced Accuracy: %.2f%%" % (balanced_accuracy_score(y_groundtruths, y_preds) * 100.0))

cm = confusion_matrix(y_groundtruths, y_preds)
plt.figure(figsize=(2,2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xticks(np.arange(2) + 0.5,[0,1])
plt.yticks(np.arange(2) + 0.5, [0,1])
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

print(classification_report(y_groundtruths, y_preds))

fpr, tpr, thresholds = roc_curve(y_groundtruths, y_pred_probs)

plt.figure(figsize=(3,3))
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr, tpr, label='XGBoost')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'XGBoost ROC Curve - AUC: {roc_auc_score(y_groundtruths, y_pred_probs):.2f}')
plt.show()

print(f'Balanced Accuracy @80sens: {calculate_bac(y_groundtruths, y_pred_probs, 0.8)[0]:.2f}')

## Frequency specific features

In [ ]:
unique_subjects=np.unique(description['subject'])

print(f'Seed: {seed}')

labels=np.where(description['epilepsy']==1,1,0)

y_groundtruths=[]
y_preds=[]
y_pred_probs=[]

for i,subject in enumerate(unique_subjects):

    test_idxs=np.where(description['subject']==subject)
    train_idxs=np.where(description['subject']!=subject)

    print(f'Iteration {i} - Subject left out: {subject}')
    print(f'Number of train samples: {len(train_idxs[0])}')
    print(f'Number of test samples: {len(test_idxs[0])}')

    x_train=feats[train_idxs]
    x_test=feats[test_idxs]
    y_train=labels[train_idxs]
    y_test=labels[test_idxs]

    ratio=len(np.where(y_train==0)[0])/len(np.where(y_train==1)[0])

    ratio=(len(y_train)-sum(y_train))/sum(y_train)                   #len(np.where(y_train==0)[0])/len(np.where(y_train==1)[0])
    model=XGBClassifier(scale_pos_weight=ratio, n_jobs=4, device='cuda:0', n_estimators=100, seed=seed, max_depth=6, subsample=0.9, gamma=0.1, learning_rate=0.01)
    
    model.fit(x_train,y_train)
    
    y_pred=model.predict(x_test)
    y_pred_prob = model.predict_proba(x_test)[:,1]

    y_groundtruths.extend(y_test)
    y_preds.extend(y_pred)
    y_pred_probs.extend(y_pred_prob)
    
accuracy = accuracy_score(y_groundtruths, y_preds)
print("Accuracy: %.2f%%" % (accuracy * 100.0))
print("Balanced Accuracy: %.2f%%" % (balanced_accuracy_score(y_groundtruths, y_preds) * 100.0))

cm = confusion_matrix(y_groundtruths, y_preds)
plt.figure(figsize=(2,2))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xticks(np.arange(2) + 0.5,[0,1])
plt.yticks(np.arange(2) + 0.5, [0,1])
plt.gca().invert_yaxis()
plt.gca().invert_xaxis()
plt.xlabel("Predicted labels")
plt.ylabel("True labels")
plt.show()

print(classification_report(y_groundtruths, y_preds))

fpr, tpr, thresholds = roc_curve(y_groundtruths, y_pred_probs)

plt.figure(figsize=(3,3))
plt.plot([0, 1], [0, 1], 'k--')
plt.plot(fpr, tpr, label='XGBoost')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title(f'XGBoost ROC Curve - AUC: {roc_auc_score(y_groundtruths, y_pred_probs):.2f}')
plt.show()

print(f'Balanced Accuracy @80sens: {calculate_bac(y_groundtruths, y_pred_probs, 0.8)[0]:.2f}')

In [1]:
print('madonna')

madonna
